# Домашнее задание №5
Задание описано в файле ДЗ 5.pdf

In [4]:
from datetime import datetime
import re
from typing import List, Dict

## Задание №1

In [5]:
class Account:
    _account_counter = 100000

    def __init__(self, account_holder: str, balance: float = 0):
        if not self._validate_holder_name(account_holder):
            raise ValueError("Имя должно быть соотвествовать шаблону 'Имя Фамилия'")

        self.__class__._account_counter += 1
        self.holder = account_holder
        self.account_number = f"ACC-{self.__class__._account_counter:06d}"
        self._balance = balance
        self.operation_history = []

    def _add_to_history(self, operation: str, amount: float, balance_after: float, status: str):
        result = {
            'Тип операции': operation,
            'Сумма пополнения': amount,
            'Дата и время операции': datetime.now(),
            'Текущий баланс:': balance_after,
            'Статус': status
        }
        self.operation_history.append(result)

    def get_balance(self) -> float:
        return self._balance

    def get_history(self) -> List[Dict]:
        return self.operation_history.copy()

    def deposit(self, amount: float):
        if amount <= 0:
            raise ValueError("Сумма пополнения должна быть положительной!")
        self._balance += amount
        self._add_to_history('deposit', amount, self._balance, 'success')

    def withdraw(self, amount: float):
        if amount <= 0:
            raise ValueError("Сумма снятия должна быть положительной!")
        if amount > self._balance:
            self._add_to_history('withdraw', amount, self._balance, 'fail')
            return
        self._balance -= amount
        self._add_to_history('withdraw', amount, self._balance, 'success')

    # От задания №2
    def _validate_holder_name(self, name: str) -> bool:
        pattern = r'^[A-ZА-Я][a-zа-я]+ [A-ZА-Я][a-zа-я]+$'
        return bool(re.match(pattern, name.strip()))

    def get_big_operations(self, n: int = 10) -> List[Dict]:
        valid = []
        for op in self.operation_history:
            if op['status'] == 'success':
                valid.append(op)
        sort = sorted(valid, reverse=True)
        return sort[:n]

## Задание №2

In [ ]:
class CheckingAccount(Account):
    account_type = 'checking'

    def __init__(self, account_holder: str, balance: float = 0):
        super().__init__(account_holder, balance)

class SavingAccount(Account):
    account_type = 'saving'

    def __init__(self, account_holder: str, balance: float = 0):
        super().__init__(account_holder, balance)

    def apply_interest(self, rate: float):
        if rate < 0:
            raise ValueError("Ставка не может быть отрицательной!")
        interest = self._balance + (rate / 100)
        self._balance += interest

    def withdraw(self, amount: float):
        if amount <= 0:
            raise ValueError("Сумма снятия должна быть положительной!")
        if amount > self._balance * 0.5:
            self._add_to_history('withdraw', amount, self._balance, 'fail')
            return
        self._balance -= amount
        self._add_to_history('withdraw', amount, self._balance, 'success')